# 10 — Full Feature Rebuild + Append New 2026 Matches

Recomputes all features from scratch (2004–present) using the same pipeline as notebook 05,  
with detailed validation at every step.  
Output: `data/processed/updated_model_dataset.csv` (full rebuild, all years).

In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path('../data/processed')
RAW_DIR       = Path('../data/raw')
OUTPUT_PATH   = PROCESSED_DIR / 'updated_model_dataset.csv'
YEAR_CUTOFF   = 2004
WINDOW        = 5
CUTOFF_DATE   = pd.Timestamp('2026-05-16')   # original model_dataset end date

TEAM_NAME_MAP = {
    'USA':                   'United States',
    'Turkey':                'Turkiye',
    'Czech Republic':        'Czechia',
    "Cote d'Ivoire":         'Ivory Coast',
    'Korea Republic':        'South Korea',
    'Cape Verde Islands':    'Cape Verde',
    'Curacao':               'Curaçao',
    'Macedonia':             'North Macedonia',
    'Swaziland':             'Eswatini',
    'Sao Tome and Principe': 'São Tomé and Príncipe',
}
print('Setup done')

Setup done


---
## SECTION 1 — RAW DATA

In [21]:
# ── Load all raw ELO files ────────────────────────────────────────────────────
elo_files = sorted(RAW_DIR.glob('elo_*_results.csv'))
print(f'Raw files found: {len(elo_files)}')
for f in elo_files:
    tmp = pd.read_csv(f)
    print(f'  {f.name:<30} {len(tmp):>5} rows')

dfs = []
for f in elo_files:
    tmp = pd.read_csv(f)
    tmp['source_file'] = f.name
    dfs.append(tmp)

raw = pd.concat(dfs, ignore_index=True)
raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
raw = raw.dropna(subset=['date'])
raw = raw[raw['date'].dt.year >= YEAR_CUTOFF].copy()
raw = raw.sort_values('date').reset_index(drop=True)

print(f'\nAfter {YEAR_CUTOFF} filter: {len(raw):,} rows')
print(f'Date range: {raw["date"].min().date()} → {raw["date"].max().date()}')
print(f'Unique teams: {pd.concat([raw["team_a"], raw["team_b"]]).nunique()}')
print(f'Unique competitions: {raw["competition"].nunique()}')

Raw files found: 26
  elo_2001_results.csv            1015 rows
  elo_2002_results.csv             779 rows
  elo_2003_results.csv             927 rows
  elo_2004_results.csv            1092 rows
  elo_2005_results.csv             793 rows
  elo_2006_results.csv             843 rows
  elo_2007_results.csv             988 rows
  elo_2008_results.csv            1114 rows
  elo_2009_results.csv             905 rows
  elo_2010_results.csv             881 rows
  elo_2011_results.csv            1104 rows
  elo_2012_results.csv            1036 rows
  elo_2013_results.csv             954 rows
  elo_2014_results.csv             881 rows
  elo_2015_results.csv            1019 rows
  elo_2016_results.csv             925 rows
  elo_2017_results.csv             886 rows
  elo_2018_results.csv             898 rows
  elo_2019_results.csv            1151 rows
  elo_2020_results.csv             366 rows
  elo_2021_results.csv            1133 rows
  elo_2022_results.csv             988 rows
  elo_2023_r

In [22]:
# ── Check for duplicates in raw data ─────────────────────────────────────────
dupes_raw = raw[raw.duplicated(subset=['date','team_a','team_b'], keep=False)]
print(f'Duplicate (date, team_a, team_b) rows in raw data: {len(dupes_raw)}')
if len(dupes_raw):
    print(dupes_raw[['date','team_a','team_b','goals_a','goals_b','competition','source_file']].to_string())

# Drop duplicates, keep first occurrence
raw = raw.drop_duplicates(subset=['date','team_a','team_b'], keep='first').reset_index(drop=True)
print(f'After dedup: {len(raw):,} rows')

Duplicate (date, team_a, team_b) rows in raw data: 6
            date    team_a          team_b  goals_a  goals_b competition           source_file
5898  2010-03-28      Saba  Sint Eustatius        2        2    Friendly  elo_2010_results.csv
5899  2010-03-28      Saba  Sint Eustatius        1        2    Friendly  elo_2010_results.csv
17092 2022-03-08  Eswatini         Lesotho        0        0    Friendly  elo_2022_results.csv
17093 2022-03-08  Eswatini         Lesotho        4        0    Friendly  elo_2022_results.csv
18193 2023-06-03  Eswatini         Lesotho        1        2    Friendly  elo_2023_results.csv
18194 2023-06-03  Eswatini         Lesotho        1        0    Friendly  elo_2023_results.csv
After dedup: 21,655 rows


In [23]:
# ── Raw data sample and structure ─────────────────────────────────────────────
print('RAW DATA COLUMNS:', list(raw.columns))
print('\nFirst 5 rows:')
display(raw.head())
print('\nLast 5 rows:')
display(raw.tail())
print('\nCompetition distribution (top 15):')
print(raw['competition'].value_counts().head(15))
print('\nMatches per year:')
print(raw['date'].dt.year.value_counts().sort_index())

RAW DATA COLUMNS: ['date', 'team_a', 'team_b', 'goals_a', 'goals_b', 'competition', 'location', 'rating_change_a', 'rating_change_b', 'rating_a', 'rating_b', 'rank_change_a', 'rank_change_b', 'rank_a', 'rank_b', 'source_file']

First 5 rows:


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
0,2004-01-01,Bermuda,Barbados,0,4,Dudley Eve Memorial Trophy,Bermuda,-28,28,1242,1397,-8,7,146,109,elo_2004_results.csv
1,2004-01-01,Kuwait,Yemen,4,0,Gulf Cup,Kuwait,6,-6,1525,1173,2,-1,72,160,elo_2004_results.csv
2,2004-01-01,Saudi Arabia,Bahrain,1,0,Gulf Cup,Kuwait,12,-12,1674,1511,3,-6,44,78,elo_2004_results.csv
3,2004-01-03,Bahrain,Oman,1,0,Gulf Cup,Kuwait,23,-23,1534,1537,7,-4,71,70,elo_2004_results.csv
4,2004-01-03,Qatar,United Arab Emirates,0,0,Gulf Cup,Kuwait,-4,4,1517,1459,-4,1,77,90,elo_2004_results.csv



Last 5 rows:


,date,team_a,team_b,goals_a,goals_b,competition,location,rating_change_a,rating_change_b,rating_a,rating_b,rank_change_a,rank_change_b,rank_a,rank_b,source_file
21650,2026-06-07,Liechtenstein,Cyprus,0,2,Friendly,Liechtenstein,-4,4,895,1314,-1,2,199,131,elo_2026_results.csv
21651,2026-06-07,Morocco,Norway,1,1,Friendly,the United States,3,-3,1827,1914,0,0,24,11,elo_2026_results.csv
21652,2026-06-07,Oman,Mozambique,4,1,Garuda Championship Series,Indonesia,20,-20,1479,1352,4,-9,83,118,elo_2026_results.csv
21653,2026-06-07,Kenya,Lesotho,4,0,Friendly,South Africa,11,-11,1363,1198,5,-3,113,159,elo_2026_results.csv
21654,2026-06-08,Sri Lanka,Bhutan,4,1,Friendly,Thailand,9,-9,836,622,0,-2,205,230,elo_2026_results.csv



Competition distribution (top 15):
competition
Friendly                               7063
World Cup qualifier                    4169
European Championship qualifier        1322
African Nations Cup qualifier          1199
African Nations Cup                     461
World Cup and Asian Cup qualifier       454
Asian Cup qualifier                     390
World Cup and African Cup qualifier     345
COSAFA Cup                              335
World Cup                               320
CECAFA Cup                              310
CONCACAF Championship                   300
European Championship                   246
Southeast Asian Championship            239
Asian Cup                               230
Name: count, dtype: int64

Matches per year:
date
2004    1092
2005     793
2006     843
2007     988
2008    1114
2009     905
2010     880
2011    1104
2012    1036
2013     954
2014     881
2015    1019
2016     925
2017     886
2018     898
2019    1151
2020     366
2021    1133
2022    

---
## SECTION 2 — PRE-MATCH VALUE RECONSTRUCTION

The raw ELO file stores **post-match** ratings and ranks.  
Pre-match values = post-match − change.

In [24]:
df = raw.copy()

df['rating_a_before'] = df['rating_a'] - df['rating_change_a']
df['rating_b_before'] = df['rating_b'] - df['rating_change_b']
df['rank_a_before']   = df['rank_a']   - df['rank_change_a']
df['rank_b_before']   = df['rank_b']   - df['rank_change_b']

df['elo_diff']  = df['rating_a_before'] - df['rating_b_before']   # positive = team_a stronger
df['rank_diff'] = df['rank_a_before']   - df['rank_b_before']     # positive = team_a ranked lower (worse)

df['target_goals_a']     = df['goals_a'].astype(int)
df['target_goals_b']     = df['goals_b'].astype(int)
df['target_goal_diff']   = df['target_goals_a'] - df['target_goals_b']
df['target_total_goals'] = df['target_goals_a'] + df['target_goals_b']

df['result_a'] = np.where(df['goals_a'] > df['goals_b'], 3,
                 np.where(df['goals_a'] == df['goals_b'], 1, 0))
df['result_b'] = np.where(df['goals_b'] > df['goals_a'], 3,
                 np.where(df['goals_a'] == df['goals_b'], 1, 0))

df['season_id']       = df['date'].dt.year
df['tournament_year'] = df['date'].dt.year
df['tournament_key']  = df['competition'].astype(str) + '_' + df['tournament_year'].astype(str)

ELO_BASE = pd.concat([df['rating_a_before'], df['rating_b_before']]).mean()
print(f'ELO_BASE (mean of all pre-match ratings): {ELO_BASE:.4f}')

print('\n── Pre-match reconstruction example (first 3 rows) ──')
example_cols = ['date','team_a','team_b',
                'rating_a','rating_change_a','rating_a_before',
                'rating_b','rating_change_b','rating_b_before',
                'elo_diff','rank_diff']
display(df[example_cols].head(3))

print('\n── ELO diff distribution ──')
print(df['elo_diff'].describe().round(2))
print('\n── Rank diff distribution ──')
print(df['rank_diff'].describe().round(2))

ELO_BASE (mean of all pre-match ratings): 1480.8046

── Pre-match reconstruction example (first 3 rows) ──


,date,team_a,team_b,rating_a,rating_change_a,rating_a_before,rating_b,rating_change_b,rating_b_before,elo_diff,rank_diff
0,2004-01-01,Bermuda,Barbados,1242,-28,1270,1397,28,1369,-99,52
1,2004-01-01,Kuwait,Yemen,1525,6,1519,1173,-6,1179,340,-91
2,2004-01-01,Saudi Arabia,Bahrain,1674,12,1662,1511,-12,1523,139,-43



── ELO diff distribution ──
count    21655.00
mean        52.25
std        278.66
min      -1238.00
25%       -119.00
50%         53.00
75%        226.00
max       1234.00
Name: elo_diff, dtype: float64

── Rank diff distribution ──
count    21655.00
mean       -12.20
std         55.04
min       -204.00
25%        -45.00
50%        -13.00
75%         19.00
max        204.00
Name: rank_diff, dtype: float64


---
## SECTION 3 — ROLLING WINDOW FEATURES

For each match, look at each team's **last 5 games** and compute:
- `weighted_goals_for`   = goals_scored × (opponent_elo / ELO_BASE)  — scoring against strong teams counts more
- `weighted_goals_against` = goals_conceded × (ELO_BASE / opponent_elo) — conceding to weak teams penalised more
- `opponent_elo`         = opponent pre-match ELO (measures quality of recent schedule)
- `rating_change`        = ELO points gained/lost (recent form signal)

All four are averaged over last 5 games. Team A − Team B diffs are the actual features.

In [25]:
def mean_or_zero(values):
    return float(np.mean(values)) if values else 0.0

def mean_or_base(values, base):
    return float(np.mean(values)) if values else float(base)

team_history = {}   # team → full chronological list of per-game dicts
rolling_records = []

VERBOSE_TEAM = 'Brazil'
verbose_all = []   # ALL Brazil appearances (team_a OR team_b), for trace validation

for idx, row in df.iterrows():
    team_a = row['team_a']
    team_b = row['team_b']
    date   = row['date']
    elo_a  = max(float(row['rating_a_before']), 1)
    elo_b  = max(float(row['rating_b_before']), 1)

    hist_a_all = team_history.get(team_a, [])
    hist_b_all = team_history.get(team_b, [])
    hist_a = hist_a_all[-WINDOW:]
    hist_b = hist_b_all[-WINDOW:]

    wgf_a  = mean_or_zero([x['weighted_goals_for']     for x in hist_a])
    wgf_b  = mean_or_zero([x['weighted_goals_for']     for x in hist_b])
    wga_a  = mean_or_zero([x['weighted_goals_against'] for x in hist_a])
    wga_b  = mean_or_zero([x['weighted_goals_against'] for x in hist_b])
    opp_a  = mean_or_base([x['opponent_elo']           for x in hist_a], ELO_BASE)
    opp_b  = mean_or_base([x['opponent_elo']           for x in hist_b], ELO_BASE)
    rch_a  = mean_or_zero([x['rating_change']          for x in hist_a])
    rch_b  = mean_or_zero([x['rating_change']          for x in hist_b])

    days_a = int(np.clip((date - hist_a_all[-1]['date']).days, 0, 60)) if hist_a_all else 60
    days_b = int(np.clip((date - hist_b_all[-1]['date']).days, 0, 60)) if hist_b_all else 60

    rolling_records.append({
        'team_a_weighted_goals_for_avg_last5':     wgf_a,
        'team_b_weighted_goals_for_avg_last5':     wgf_b,
        'team_a_weighted_goals_against_avg_last5': wga_a,
        'team_b_weighted_goals_against_avg_last5': wga_b,
        'team_a_avg_opponent_elo_last5':           opp_a,
        'team_b_avg_opponent_elo_last5':           opp_b,
        'team_a_rating_change_avg_last5':          rch_a,
        'team_b_rating_change_avg_last5':          rch_b,
        'team_a_matches_played_before':            len(hist_a_all),
        'team_b_matches_played_before':            len(hist_b_all),
        'team_a_days_since_last_match':            days_a,
        'team_b_days_since_last_match':            days_b,
    })

    # Capture ALL appearances of VERBOSE_TEAM (both as team_a AND team_b)
    if team_a == VERBOSE_TEAM or team_b == VERBOSE_TEAM:
        is_home      = (team_a == VERBOSE_TEAM)
        v_hist_all   = hist_a_all if is_home else hist_b_all
        v_hist5      = hist_a     if is_home else hist_b
        v_wgf        = wgf_a      if is_home else wgf_b
        v_opp        = opp_a      if is_home else opp_b
        v_goals_for  = int(row['target_goals_a']) if is_home else int(row['target_goals_b'])
        v_goals_ag   = int(row['target_goals_b']) if is_home else int(row['target_goals_a'])
        v_opp_elo    = elo_b if is_home else elo_a
        verbose_all.append({
            'date':        date,
            'role':        'A' if is_home else 'B',
            'opponent':    team_b if is_home else team_a,
            'gf':          v_goals_for,
            'ga':          v_goals_ag,
            'opp_elo':     round(v_opp_elo, 0),
            'history_len': len(v_hist_all),      # games in history BEFORE this match
            'last5':       [(round(x['weighted_goals_for'], 3), round(x['opponent_elo'], 0))
                            for x in v_hist5],
            'wgf_avg':     round(v_wgf, 4),
            'opp_elo_avg': round(v_opp, 1),
        })

    # Update history AFTER recording pre-match features
    team_history.setdefault(team_a, []).append({
        'date':                   date,
        'weighted_goals_for':     row['target_goals_a'] * (elo_b / ELO_BASE),
        'weighted_goals_against': row['target_goals_b'] * (ELO_BASE / elo_b),
        'opponent_elo':           elo_b,
        'rating_change':          float(row['rating_change_a']),
    })
    team_history.setdefault(team_b, []).append({
        'date':                   date,
        'weighted_goals_for':     row['target_goals_b'] * (elo_a / ELO_BASE),
        'weighted_goals_against': row['target_goals_a'] * (ELO_BASE / elo_a),
        'opponent_elo':           elo_a,
        'rating_change':          float(row['rating_change_b']),
    })

rolling_df = pd.DataFrame(rolling_records)
df = pd.concat([df.reset_index(drop=True), rolling_df], axis=1)

df['weighted_goals_for_diff_last5']     = df['team_a_weighted_goals_for_avg_last5']     - df['team_b_weighted_goals_for_avg_last5']
df['weighted_goals_against_diff_last5'] = df['team_a_weighted_goals_against_avg_last5'] - df['team_b_weighted_goals_against_avg_last5']
df['opponent_strength_diff_last5']      = df['team_a_avg_opponent_elo_last5']            - df['team_b_avg_opponent_elo_last5']
df['rating_change_diff_last5']          = df['team_a_rating_change_avg_last5']           - df['team_b_rating_change_avg_last5']

print(f'Rolling features computed for {len(df):,} rows')
print(f'{VERBOSE_TEAM} total appearances (both roles): {len(verbose_all)}')

Rolling features computed for 21,655 rows
Brazil total appearances (both roles): 306


In [26]:
# ── Full chronological trace for VERBOSE_TEAM (first 15 games, both roles) ────
# role=A means Brazil was team_a in that row; role=B means Brazil was team_b.
# history_len = number of games tracked before this match (both roles count).
# The last5 shows the wGF values and opponent ELOs that fed into the rolling average.

print(f'=== {VERBOSE_TEAM} — chronological appearance trace (first 15 games) ===')
print(f'{"#":<3} {"Date":<12} {"R":<2} {"Opponent":<22} {"Score":<6} {"OppELO":>7} '
      f'{"HistLen":>7} {"last5_wGF_values":<40} {"wGF_avg":>8}')
print('-' * 115)

for i, ex in enumerate(verbose_all[:15]):
    last5_str = str([(wgf, elo) for wgf, elo in ex['last5']]) if ex['last5'] else '[]'
    print(f'{i+1:<3} {str(ex["date"].date()):<12} {ex["role"]:<2} {ex["opponent"]:<22} '
          f'{ex["gf"]}–{ex["ga"]:<4} {ex["opp_elo"]:>7.0f} '
          f'{ex["history_len"]:>7} {last5_str:<40} {ex["wgf_avg"]:>8.4f}')

print()
print('Key: R=A means Brazil listed as team_a, R=B means team_b.')
print('HistLen = total games in team_history before this match (both roles).')
print('wGF_avg = mean of weighted_goals_for across last 5 entries in history.')
print()
print(f'── Verify: going into game 3, does wGF_avg match manual calculation? ──')
ex3 = verbose_all[2]
manual = sum(wgf for wgf, _ in ex3['last5']) / len(ex3['last5']) if ex3['last5'] else 0
print(f'  last5 wGF values: {[wgf for wgf, _ in ex3["last5"]]}')
print(f'  manual mean:  {manual:.4f}')
print(f'  stored avg:   {ex3["wgf_avg"]:.4f}')
print(f'  match: {abs(manual - ex3["wgf_avg"]) < 1e-9}')

=== Brazil — chronological appearance trace (first 15 games) ===
#   Date         R  Opponent               Score   OppELO HistLen last5_wGF_values                          wGF_avg
-------------------------------------------------------------------------------------------------------------------
1   2004-02-18   B  Ireland                0–0       1833       0 []                                         0.0000
2   2004-03-31   B  Paraguay               0–0       1792       1 [(0.0, 1833.0)]                            0.0000
3   2004-04-28   B  Hungary                4–1       1556       2 [(0.0, 1833.0), (0.0, 1792.0)]             0.0000
4   2004-05-20   B  France                 0–0       2059       3 [(0.0, 1833.0), (0.0, 1792.0), (4.203, 1556.0)]   1.4010
5   2004-06-02   A  Argentina              3–1       2026       4 [(0.0, 1833.0), (0.0, 1792.0), (4.203, 1556.0), (0.0, 2059.0)]   1.0508
6   2004-06-06   B  Chile                  1–1       1693       5 [(0.0, 1833.0), (0.0, 1792.0

In [27]:
# ── Manual verification: spot-check one specific match ───────────────────────
# Pick Brazil's first WC 2022 match and verify the rolling feature by hand
wc2022_brazil = df[(df['team_a'] == 'Brazil') & (df['tournament_key'] == 'World Cup_2022')]
if len(wc2022_brazil):
    row_check = wc2022_brazil.iloc[0]
    team = 'Brazil'
    match_date = row_check['date']
    print(f'Spot-check: {team} at WC 2022, first match on {match_date.date()}')
    print(f'  Opponent: {row_check["team_b"]}')
    print(f'  Score: {int(row_check["target_goals_a"])}–{int(row_check["target_goals_b"])}')
    print(f'  elo_diff (pre-match): {row_check["elo_diff"]}')
    print(f'  rating_a_before: {row_check["rating_a_before"]}   rating_b_before: {row_check["rating_b_before"]}')
    print(f'  weighted_goals_for_avg_last5 (team_a): {row_check["team_a_weighted_goals_for_avg_last5"]:.4f}')
    print(f'  avg_opponent_elo_last5 (team_a):       {row_check["team_a_avg_opponent_elo_last5"]:.1f}')
    print(f'  matches_played_before (team_a):        {row_check["team_a_matches_played_before"]}')
    print(f'  days_since_last_match (team_a):        {row_check["team_a_days_since_last_match"]}')

    # Now manually fetch Brazil's last 5 games before this date from team_history
    brazil_hist = team_history.get(team, [])
    before = [x for x in brazil_hist if x['date'] < match_date]
    last5  = before[-5:]
    print(f'\n  Manual last-5 from team_history:')
    for g in last5:
        print(f"    {g['date'].date()}  wGF={g['weighted_goals_for']:.3f}  wGA={g['weighted_goals_against']:.3f}  oppELO={g['opponent_elo']:.0f}  rchg={g['rating_change']:+.0f}")
    manual_wgf = np.mean([g['weighted_goals_for'] for g in last5]) if last5 else 0
    print(f'  Manual mean wGF: {manual_wgf:.4f}  ← should match {row_check["team_a_weighted_goals_for_avg_last5"]:.4f}')

Spot-check: Brazil at WC 2022, first match on 2022-11-24
  Opponent: Serbia
  Score: 2–0
  elo_diff (pre-match): 271
  rating_a_before: 2169   rating_b_before: 1898
  weighted_goals_for_avg_last5 (team_a): 4.1079
  avg_opponent_elo_last5 (team_a):       1694.0
  matches_played_before (team_a):        264
  days_since_last_match (team_a):        58

  Manual last-5 from team_history:
    2022-03-29  wGF=4.433  wGA=0.000  oppELO=1641  rchg=+7
    2022-06-02  wGF=6.081  wGA=0.822  oppELO=1801  rchg=+7
    2022-06-06  wGF=1.220  wGA=0.000  oppELO=1806  rchg=+4
    2022-09-23  wGF=3.110  wGA=0.000  oppELO=1535  rchg=+1
    2022-09-27  wGF=5.696  wGA=0.878  oppELO=1687  rchg=+2
  Manual mean wGF: 4.1079  ← should match 4.1079


In [28]:
# ── Rolling feature distributions ─────────────────────────────────────────────
rolling_feat_cols = [
    'weighted_goals_for_diff_last5',
    'weighted_goals_against_diff_last5',
    'opponent_strength_diff_last5',
    'rating_change_diff_last5',
    'team_a_matches_played_before',
    'team_a_days_since_last_match',
]
print('Rolling feature distributions:')
display(df[rolling_feat_cols].describe().round(4))

Rolling feature distributions:


,weighted_goals_for_diff_last5,weighted_goals_against_diff_last5,opponent_strength_diff_last5,rating_change_diff_last5,team_a_matches_played_before,team_a_days_since_last_match
count,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000
mean,0.1006,-0.1597,17.9232,0.0336,117.6861,24.3904
std,0.9247,1.2546,185.8821,10.0142,80.5288,23.4772
min,-4.8149,-16.8103,-884.6000,-62.6000,0.0000,0.0000
25%,-0.4647,-0.7027,-87.2000,-6.4000,50.0000,4.0000
50%,0.0955,-0.0859,9.4000,0.2000,106.0000,9.0000
75%,0.6634,0.4801,115.6000,6.4000,175.0000,52.0000
max,5.4578,13.6997,1129.0000,49.5000,406.0000,60.0000


---
## SECTION 4 — TOURNAMENT STATE FEATURES

Within each `tournament_key` (competition + year), accumulate match-by-match:  
- points (3/1/0), goals_for, goals_against, goal_diff  

Features recorded **before** each match (pre-match state):
- `team_a_tournament_matches_played`
- `tournament_points_diff` = team_a_points − team_b_points
- `tournament_goal_diff_diff` = team_a_goal_diff − team_b_goal_diff

In [29]:
def empty_state():
    return {'matches': 0, 'points': 0, 'goals_for': 0, 'goals_against': 0, 'goal_diff': 0}

tournament_states = {}
ts_records = []

# Verbose trace for 2022 WC
verbose_wc2022 = []

for _, row in df.iterrows():
    team_a  = row['team_a']
    team_b  = row['team_b']
    goals_a = int(row['target_goals_a'])
    goals_b = int(row['target_goals_b'])
    t_key   = row['tournament_key']

    tournament_states.setdefault(t_key, {})
    tournament_states[t_key].setdefault(team_a, empty_state())
    tournament_states[t_key].setdefault(team_b, empty_state())

    sa = tournament_states[t_key][team_a]
    sb = tournament_states[t_key][team_b]

    # Record pre-match state as features
    ts_records.append({
        'team_a_tournament_matches_played': sa['matches'],
        'team_b_tournament_matches_played': sb['matches'],
        'tournament_points_diff':           sa['points']    - sb['points'],
        'tournament_goal_diff_diff':        sa['goal_diff'] - sb['goal_diff'],
    })

    # Capture first few WC 2022 matches for verbose display
    if t_key == 'World Cup_2022' and len(verbose_wc2022) < 6:
        verbose_wc2022.append({
            'date': row['date'].date(), 'team_a': team_a, 'team_b': team_b,
            'score': f"{goals_a}–{goals_b}",
            'pre_pts_a': sa['points'], 'pre_pts_b': sb['points'],
            'pre_gd_a': sa['goal_diff'], 'pre_gd_b': sb['goal_diff'],
            'pre_played_a': sa['matches'], 'pre_played_b': sb['matches'],
        })

    # Update post-match
    ra = int(row['result_a']); rb = int(row['result_b'])
    sa['matches'] += 1; sa['points'] += ra
    sa['goals_for'] += goals_a; sa['goals_against'] += goals_b
    sa['goal_diff'] = sa['goals_for'] - sa['goals_against']
    sb['matches'] += 1; sb['points'] += rb
    sb['goals_for'] += goals_b; sb['goals_against'] += goals_a
    sb['goal_diff'] = sb['goals_for'] - sb['goals_against']

ts_df = pd.DataFrame(ts_records)
df = pd.concat([df.reset_index(drop=True), ts_df], axis=1)
print('Tournament state features computed')

Tournament state features computed


In [30]:
# ── Verbose example: first 6 WC 2022 matches ──────────────────────────────────
print('=== Tournament state trace — 2022 World Cup (first 6 matches) ===')
print(f'{"Date":<12} {"Team A":<22} {"Team B":<22} {"Score":<6} {"PtsA":>5} {"PtsB":>5} {"GDA":>5} {"GDB":>5} {"PlA":>4} {"PlB":>4}')
print('-' * 95)
for g in verbose_wc2022:
    print(f'{str(g["date"]):<12} {g["team_a"]:<22} {g["team_b"]:<22} {g["score"]:<6} '
          f'{g["pre_pts_a"]:>5} {g["pre_pts_b"]:>5} {g["pre_gd_a"]:>5} {g["pre_gd_b"]:>5} '
          f'{g["pre_played_a"]:>4} {g["pre_played_b"]:>4}')
print('\n(PtsA/B = pre-match tournament points, GDA/B = pre-match goal diff, PlA/B = matches played before)')

=== Tournament state trace — 2022 World Cup (first 6 matches) ===
Date         Team A                 Team B                 Score   PtsA  PtsB   GDA   GDB  PlA  PlB
-----------------------------------------------------------------------------------------------
2022-11-20   Qatar                  Ecuador                0–2        0     0     0     0    0    0
2022-11-21   Netherlands            Senegal                2–0        0     0     0     0    0    0
2022-11-21   England                Iran                   6–2        0     0     0     0    0    0
2022-11-21   United States          Wales                  1–1        0     0     0     0    0    0
2022-11-22   Saudi Arabia           Argentina              2–1        0     0     0     0    0    0
2022-11-22   Mexico                 Poland                 0–0        0     0     0     0    0    0

(PtsA/B = pre-match tournament points, GDA/B = pre-match goal diff, PlA/B = matches played before)


In [31]:
# ── Tournament feature distributions ─────────────────────────────────────────
ts_cols = ['team_a_tournament_matches_played', 'team_b_tournament_matches_played',
           'tournament_points_diff', 'tournament_goal_diff_diff']
print('Tournament state feature distributions:')
display(df[ts_cols].describe().round(3))
print('\nMatches where tournament_matches_played > 0 (team already played in tournament):')
print(f'  team_a: {(df["team_a_tournament_matches_played"] > 0).sum():,} rows ({(df["team_a_tournament_matches_played"] > 0).mean()*100:.1f}%)')

Tournament state feature distributions:


,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff
count,21655.000,21655.000,21655.000,21655.000
mean,2.025,1.970,0.254,0.378
std,2.025,1.956,3.736,5.570
min,0.000,0.000,-28.000,-65.000
25%,0.000,0.000,-1.000,-1.000
50%,1.000,1.000,0.000,0.000
75%,3.000,3.000,2.000,2.000
max,17.000,15.000,37.000,46.000



Matches where tournament_matches_played > 0 (team already played in tournament):
  team_a: 15,967 rows (73.7%)


---
## SECTION 5 — MARKET VALUE JOIN (Transfermarkt)

In [32]:
tm = pd.read_csv(PROCESSED_DIR / 'transfermarkt_market_values_clean.csv')
print(f'Transfermarkt: {len(tm):,} rows, seasons {tm["season_id"].min()}–{tm["season_id"].max()}')
print(f'Teams in TM: {tm["team_name_tm"].nunique()}')
print(f'Non-zero market value rows: {(tm["squad_market_value_millions_eur"] > 0).sum():,} ({(tm["squad_market_value_millions_eur"] > 0).mean()*100:.1f}%)')
print('\nSample TM data (Brazil):')
display(tm[tm['team_name_tm'] == 'Brazil'][['team_name_tm','season_id','avg_player_value_millions_eur','market_value_relative_to_year_mean','log_market_value']].tail(5))

Transfermarkt: 5,014 rows, seasons 2004–2026
Teams in TM: 218
Non-zero market value rows: 3,798 (75.7%)

Sample TM data (Brazil):


,team_name_tm,season_id,avg_player_value_millions_eur,market_value_relative_to_year_mean,log_market_value
616,Brazil,2022,30.58,10.195254,7.158903
617,Brazil,2023,29.04,12.716154,7.358385
618,Brazil,2024,29.11,10.467503,7.283723
619,Brazil,2025,30.74,11.156514,7.415055
620,Brazil,2026,34.82,8.232542,6.809260


In [33]:
df['team_a_tm'] = df['team_a'].replace(TEAM_NAME_MAP)
df['team_b_tm'] = df['team_b'].replace(TEAM_NAME_MAP)

mv_cols = ['team_name_tm', 'season_id',
           'avg_player_value_millions_eur',
           'market_value_relative_to_year_mean',
           'log_market_value']

tm_a = tm[mv_cols].rename(columns={
    'team_name_tm':                       'team_a_tm',
    'avg_player_value_millions_eur':       'avg_player_value_a',
    'market_value_relative_to_year_mean':  'market_value_rel_mean_a',
    'log_market_value':                   'log_market_value_a',
})
tm_b = tm[mv_cols].rename(columns={
    'team_name_tm':                       'team_b_tm',
    'avg_player_value_millions_eur':       'avg_player_value_b',
    'market_value_relative_to_year_mean':  'market_value_rel_mean_b',
    'log_market_value':                   'log_market_value_b',
})

df = df.merge(tm_a, on=['team_a_tm', 'season_id'], how='left')
df = df.merge(tm_b, on=['team_b_tm', 'season_id'], how='left')

for col in ['avg_player_value_a','avg_player_value_b',
            'market_value_rel_mean_a','market_value_rel_mean_b',
            'log_market_value_a','log_market_value_b']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

df['avg_player_value_diff']      = df['avg_player_value_a']    - df['avg_player_value_b']
df['market_value_rel_mean_diff'] = df['market_value_rel_mean_a'] - df['market_value_rel_mean_b']

cov_a = (df['avg_player_value_a'] > 0).mean() * 100
cov_b = (df['avg_player_value_b'] > 0).mean() * 100
print(f'Market value coverage  team_a: {cov_a:.1f}%   team_b: {cov_b:.1f}%')

# Show which team names did NOT match TM
elo_teams  = set(pd.concat([df['team_a'], df['team_b']]).unique())
tm_teams   = set(tm['team_name_tm'].unique())
mapped     = set(TEAM_NAME_MAP.values())
unmatched  = {t for t in elo_teams if t not in tm_teams and TEAM_NAME_MAP.get(t, t) not in tm_teams}
print(f'\nELO team names with no TM match ({len(unmatched)}): {sorted(unmatched)[:20]}')

print('\nMarket value feature distributions:')
display(df[['avg_player_value_diff','market_value_rel_mean_diff','log_market_value_a']].describe().round(4))

Market value coverage  team_a: 88.7%   team_b: 86.1%

ELO team names with no TM match (26): ['Bosnia and Herzegovina', 'Brunei', 'Chagos Islands', 'Cocos Islands', 'Congo', 'DR Congo', 'East Timor', 'Eastern Samoa', 'Falkland Islands', 'Gambia', 'Ireland', 'Kurdistan', 'Macao', 'Northern Cyprus', 'Reunion', 'Saba', 'Saint Barthelemy', 'Saint Martin', 'Saint Pierre and Miquelon', 'Sint Eustatius']

Market value feature distributions:


,avg_player_value_diff,market_value_rel_mean_diff,log_market_value_a
count,21655.0000,21655.0000,21655.0000
mean,0.6415,0.2720,2.8776
std,6.1639,2.5176,2.0891
min,-47.7800,-14.9633,0.0000
25%,-0.4200,-0.1266,0.9361
50%,0.0500,0.0078,2.8959
75%,1.0500,0.4807,4.5995
max,49.3200,14.9633,7.6408


In [34]:
# ── Example: Brazil vs Germany 2014 WC Final ──────────────────────────────────
sample = df[(df['team_a'].isin(['Brazil','Germany'])) &
            (df['team_b'].isin(['Brazil','Germany'])) &
            (df['tournament_key'] == 'World Cup_2014')]
if len(sample):
    r = sample.iloc[0]
    print(f"Market value example: {r['team_a']} vs {r['team_b']} ({r['date'].date()})")
    print(f"  avg_player_value_a: {r['avg_player_value_a']:.3f}M  avg_player_value_b: {r['avg_player_value_b']:.3f}M")
    print(f"  avg_player_value_diff: {r['avg_player_value_diff']:.3f}M  (positive = team_a richer)")
    print(f"  market_value_rel_mean_diff: {r['market_value_rel_mean_diff']:.4f}")
    print(f"  log_market_value_a: {r['log_market_value_a']:.4f}")

Market value example: Brazil vs Germany (2014-07-08)
  avg_player_value_a: 13.040M  avg_player_value_b: 18.090M
  avg_player_value_diff: -5.050M  (positive = team_a richer)
  market_value_rel_mean_diff: -1.4859
  log_market_value_a: 6.5401


---
## SECTION 6 — POSITION VALUE JOIN (goalkeeper_share_diff, defender_share_diff)

In [35]:
pv = pd.read_csv(PROCESSED_DIR / 'transfermarkt_position_values_2004_2026.csv')
print(f'Position values: {len(pv):,} rows, seasons {pv["season_id"].min()}–{pv["season_id"].max()}')
print(f'Non-zero goalkeeper rows: {(pv["goalkeeper_market_value_millions_eur"] > 0).sum():,}')
print('\nSample (Brazil 2022):')
display(pv[(pv['team_name_tm']=='Brazil') & (pv['season_id']==2022)]
        [['team_name_tm','season_id','goalkeeper_market_value_millions_eur',
          'defender_market_value_millions_eur','scraped_total_market_value_millions_eur']])

Position values: 5,014 rows, seasons 2004–2026
Non-zero goalkeeper rows: 2,013

Sample (Brazil 2022):


,team_name_tm,season_id,goalkeeper_market_value_millions_eur,defender_market_value_millions_eur,scraped_total_market_value_millions_eur
3950,Brazil,2022,0.0,0.0,0.0


In [36]:
pv_cols = ['team_name_tm', 'season_id',
           'goalkeeper_market_value_millions_eur',
           'defender_market_value_millions_eur',
           'scraped_total_market_value_millions_eur']

pv_a = pv[pv_cols].rename(columns={
    'team_name_tm':                           'team_a_tm',
    'goalkeeper_market_value_millions_eur':    'gk_val_a',
    'defender_market_value_millions_eur':      'def_val_a',
    'scraped_total_market_value_millions_eur': 'total_val_a',
})
pv_b = pv[pv_cols].rename(columns={
    'team_name_tm':                           'team_b_tm',
    'goalkeeper_market_value_millions_eur':    'gk_val_b',
    'defender_market_value_millions_eur':      'def_val_b',
    'scraped_total_market_value_millions_eur': 'total_val_b',
})

df = df.merge(pv_a, on=['team_a_tm', 'season_id'], how='left')
df = df.merge(pv_b, on=['team_b_tm', 'season_id'], how='left')

for col in ['gk_val_a','def_val_a','total_val_a','gk_val_b','def_val_b','total_val_b']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)

def safe_share(val, total):
    return np.where(total > 0, val / total, 0.0)

df['gk_share_a']  = safe_share(df['gk_val_a'],  df['total_val_a'])
df['gk_share_b']  = safe_share(df['gk_val_b'],  df['total_val_b'])
df['def_share_a'] = safe_share(df['def_val_a'],  df['total_val_a'])
df['def_share_b'] = safe_share(df['def_val_b'],  df['total_val_b'])

df['goalkeeper_share_diff'] = df['gk_share_a']  - df['gk_share_b']
df['defender_share_diff']   = df['def_share_a'] - df['def_share_b']

pv_cov_a = (df['gk_val_a'] > 0).mean() * 100
pv_cov_b = (df['gk_val_b'] > 0).mean() * 100
print(f'Position value coverage  team_a: {pv_cov_a:.1f}%   team_b: {pv_cov_b:.1f}%')

print('\nPosition share feature distributions:')
display(df[['goalkeeper_share_diff','defender_share_diff']].describe().round(5))

# Example
ex_pv = df[(df['team_a']=='Brazil') & (df['season_id']==2022)].iloc[0]
print(f"\nExample Brazil 2022: gk_val={ex_pv['gk_val_a']:.1f}  total={ex_pv['total_val_a']:.1f}  "
      f"gk_share={ex_pv['gk_share_a']:.4f}  → goalkeeper_share_diff={ex_pv['goalkeeper_share_diff']:.4f}")

Position value coverage  team_a: 50.2%   team_b: 48.0%

Position share feature distributions:


,goalkeeper_share_diff,defender_share_diff
count,21655.00000,21655.00000
mean,0.00173,0.00657
std,0.09766,0.23284
min,-1.00000,-1.00000
25%,-0.01069,-0.05785
50%,0.00000,0.00000
75%,0.01654,0.07634
max,1.00000,1.00000



Example Brazil 2022: gk_val=0.0  total=0.0  gk_share=0.0000  → goalkeeper_share_diff=0.0000


---
## SECTION 7 — ASSEMBLE FINAL DATASET

In [37]:
metadata_cols = ['date','team_a','team_b','competition','location',
                 'season_id','tournament_year','tournament_key']

feature_cols = [
    'rank_diff', 'elo_diff', 'rating_a_before', 'rating_b_before',
    'avg_player_value_diff', 'log_market_value_a', 'market_value_rel_mean_diff',
    'opponent_strength_diff_last5',
    'weighted_goals_for_diff_last5', 'weighted_goals_against_diff_last5',
    'rating_change_diff_last5',
    'team_a_matches_played_before', 'team_b_matches_played_before',
    'team_a_days_since_last_match',  'team_b_days_since_last_match',
    'team_a_tournament_matches_played', 'team_b_tournament_matches_played',
    'tournament_points_diff', 'tournament_goal_diff_diff',
    'goalkeeper_share_diff', 'defender_share_diff',
]

target_cols = ['target_goals_a', 'target_goals_b', 'target_goal_diff', 'target_total_goals']

final_cols = metadata_cols + feature_cols + target_cols

model_df = df[final_cols].copy()

# Cast integer columns
int_cols = ['season_id','tournament_year','rank_diff','elo_diff',
            'rating_a_before','rating_b_before',
            'team_a_matches_played_before','team_b_matches_played_before',
            'team_a_days_since_last_match','team_b_days_since_last_match',
            'team_a_tournament_matches_played','team_b_tournament_matches_played',
            'tournament_points_diff','tournament_goal_diff_diff',
            'target_goals_a','target_goals_b','target_goal_diff','target_total_goals']
for col in int_cols:
    model_df[col] = model_df[col].fillna(0).astype(int)

print(f'Final dataset: {len(model_df):,} rows × {len(model_df.columns)} columns')
print(f'Date range: {model_df["date"].min().date()} → {model_df["date"].max().date()}')
print(f'New rows (after {CUTOFF_DATE.date()}): {(model_df["date"] > CUTOFF_DATE).sum()}')
print(f'\nMissing values:')
missing = model_df.isnull().sum()
print(missing[missing > 0] if missing.any() else '  None')

Final dataset: 21,655 rows × 33 columns
Date range: 2004-01-01 → 2026-06-08
New rows (after 2026-05-16): 119

Missing values:
  None


---
## SECTION 8 — FULL FEATURE DISTRIBUTION OVERVIEW

In [38]:
print('=== ALL FEATURE DISTRIBUTIONS ===')
display(model_df[feature_cols].describe().round(4))

=== ALL FEATURE DISTRIBUTIONS ===


,rank_diff,elo_diff,rating_a_before,rating_b_before,avg_player_value_diff,log_market_value_a,market_value_rel_mean_diff,opponent_strength_diff_last5,weighted_goals_for_diff_last5,weighted_goals_against_diff_last5,...,team_a_matches_played_before,team_b_matches_played_before,team_a_days_since_last_match,team_b_days_since_last_match,team_a_tournament_matches_played,team_b_tournament_matches_played,tournament_points_diff,tournament_goal_diff_diff,goalkeeper_share_diff,defender_share_diff
count,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,...,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000,21655.0000
mean,-12.1986,52.2512,1506.9302,1454.6790,0.6415,2.8776,0.2720,17.9232,0.1006,-0.1597,...,117.6861,109.9175,24.3904,24.6065,2.0254,1.9703,0.2542,0.3778,0.0017,0.0066
std,55.0357,278.6552,298.8551,317.8751,6.1639,2.0891,2.5176,185.8821,0.9247,1.2546,...,80.5288,77.1307,23.4772,23.6530,2.0245,1.9555,3.7361,5.5704,0.0977,0.2328
min,-204.0000,-1238.0000,334.0000,335.0000,-47.7800,0.0000,-14.9633,-884.6000,-4.8149,-16.8103,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-28.0000,-65.0000,-1.0000,-1.0000
25%,-45.0000,-119.0000,1314.5000,1271.0000,-0.4200,0.9361,-0.1266,-87.2000,-0.4647,-0.7027,...,50.0000,45.0000,4.0000,4.0000,0.0000,0.0000,-1.0000,-1.0000,-0.0107,-0.0578
50%,-13.0000,53.0000,1529.0000,1490.0000,0.0500,2.8959,0.0078,9.4000,0.0955,-0.0859,...,106.0000,98.0000,9.0000,10.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000
75%,19.0000,226.0000,1726.0000,1685.0000,1.0500,4.5995,0.4807,115.6000,0.6634,0.4801,...,175.0000,164.0000,52.0000,55.0000,3.0000,3.0000,2.0000,2.0000,0.0165,0.0763
max,204.0000,1234.0000,2223.0000,2199.0000,49.3200,7.6408,14.9633,1129.0000,5.4578,13.6997,...,406.0000,403.0000,60.0000,60.0000,17.0000,15.0000,37.0000,46.0000,1.0000,1.0000


In [39]:
# ── Compare existing dataset vs new rows ──────────────────────────────────────
existing = pd.read_csv(PROCESSED_DIR / 'model_dataset.csv')
existing['date'] = pd.to_datetime(existing['date'])

old_rows = model_df[model_df['date'] <= CUTOFF_DATE]
new_rows = model_df[model_df['date'] >  CUTOFF_DATE]

print(f'Old rows (≤ {CUTOFF_DATE.date()}): {len(old_rows):,}')
print(f'New rows (> {CUTOFF_DATE.date()}): {len(new_rows):,}')
print(f'Existing model_dataset rows:    {len(existing):,}')

check_cols = ['elo_diff','rank_diff','weighted_goals_for_diff_last5',
              'avg_player_value_diff','goalkeeper_share_diff','defender_share_diff']

print('\nFeature means — rebuilt vs existing vs new:')
comp = pd.DataFrame({
    'rebuilt_old': old_rows[check_cols].mean(),
    'existing_dataset': existing[check_cols].mean(),
    'new_rows': new_rows[check_cols].mean(),
}).round(4)
print(comp)
print('\n(rebuilt_old vs existing_dataset should be very close — validates feature correctness)')

Old rows (≤ 2026-05-16): 21,536
New rows (> 2026-05-16): 119
Existing model_dataset rows:    21,539

Feature means — rebuilt vs existing vs new:
                               rebuilt_old  existing_dataset  new_rows
elo_diff                           51.7202           51.7137  148.3445
rank_diff                         -12.1111          -12.1110  -28.0336
weighted_goals_for_diff_last5       0.0994            0.0993    0.3112
avg_player_value_diff               0.6221            0.6220    4.1477
goalkeeper_share_diff               0.0017            0.0017    0.0000
defender_share_diff                 0.0065            0.0065    0.0165

(rebuilt_old vs existing_dataset should be very close — validates feature correctness)


In [40]:
# ── New rows detail ───────────────────────────────────────────────────────────
print(f'New rows competition breakdown:')
print(new_rows['competition'].value_counts())

print(f'\nNew rows feature distributions:')
display(new_rows[check_cols].describe().round(4))

print('\nNew rows sample (first 10):')
display(new_rows[['date','team_a','team_b','elo_diff','rank_diff',
                   'weighted_goals_for_diff_last5','avg_player_value_diff',
                   'goalkeeper_share_diff','target_goals_a','target_goals_b']].head(10))

New rows competition breakdown:
competition
Friendly                             104
Unity Cup                              4
Diamond Jubilee Tournament             3
Friendly tournament                    2
Garuda Championship Series             2
Baltic Cup                             2
Southeast Asian Championship qual      1
Asian Cup qualifier                    1
Name: count, dtype: int64

New rows feature distributions:


,elo_diff,rank_diff,weighted_goals_for_diff_last5,avg_player_value_diff,goalkeeper_share_diff,defender_share_diff
count,119.0000,119.0000,119.0000,119.0000,119.0000,119.0000
mean,148.3445,-28.0336,0.3112,4.1477,0.0000,0.0165
std,221.5121,41.7125,0.8162,10.6940,0.0940,0.2255
min,-420.0000,-134.0000,-1.6328,-25.4600,-0.2777,-0.6338
25%,-24.5000,-55.0000,-0.2752,-0.1400,-0.0491,-0.1019
50%,172.0000,-29.0000,0.3039,0.5000,0.0000,0.0057
75%,321.0000,1.5000,0.8548,6.1100,0.0482,0.1707
max,633.0000,77.0000,2.3898,49.3200,0.2943,0.6409



New rows sample (first 10):


,date,team_a,team_b,elo_diff,rank_diff,weighted_goals_for_diff_last5,avg_player_value_diff,goalkeeper_share_diff,target_goals_a,target_goals_b
21536,2026-05-22,Mexico,Ghana,352,-62,0.864395,-2.88,0.039706,2,0
21537,2026-05-26,Nigeria,Zimbabwe,381,-82,-0.357373,6.04,0.082622,2,0
21538,2026-05-26,Morocco,Burundi,535,-120,0.004187,7.50,0.000896,5,0
21539,2026-05-27,Jamaica,India,397,-95,-0.409237,1.82,-0.146497,2,0
21540,2026-05-28,Ireland,Qatar,267,-48,1.652210,-0.73,-0.090354,1,0
21541,2026-05-28,Egypt,Russia,-87,8,0.078741,-2.70,-0.072316,1,0
21542,2026-05-29,Bosnia and Herzegovina,North Macedonia,5,3,1.673279,-1.33,-0.072981,0,0
21543,2026-05-29,Iran,Gambia,337,-71,-0.646135,1.18,0.056662,3,1
21544,2026-05-29,South Africa,Nicaragua,198,-45,0.333197,1.44,-0.061345,0,0
21545,2026-05-29,Iraq,Andorra,530,-112,0.969743,0.50,-0.277704,1,0


In [41]:
# ── Duplicates check ──────────────────────────────────────────────────────────
dupes = model_df[model_df.duplicated(subset=['date','team_a','team_b'], keep=False)]
print(f'Duplicate (date, team_a, team_b) rows: {len(dupes)}')
if len(dupes):
    display(dupes[['date','team_a','team_b','target_goals_a','target_goals_b','competition']].sort_values(['date','team_a']))

Duplicate (date, team_a, team_b) rows: 0


---
## SECTION 9 — SAVE

---
## SECTION 10 — NEW ROWS FEATURE INSPECTION (post-May 16 2026)

For every one of the 20 model features: zero count, distribution, and example rows.

In [43]:
FEATURE_COLS_20 = [
    'rank_diff',
    'elo_diff',
    'rating_a_before',
    'rating_b_before',
    'avg_player_value_diff',
    'opponent_strength_diff_last5',
    'weighted_goals_for_diff_last5',
    'weighted_goals_against_diff_last5',
    'team_b_matches_played_before',
    'team_a_matches_played_before',
    'market_value_rel_mean_diff',
    'rating_change_diff_last5',
    'defender_share_diff',
    'goalkeeper_share_diff',
    'team_b_days_since_last_match',
    'team_a_days_since_last_match',
    'tournament_goal_diff_diff',
    'tournament_points_diff',
    'team_a_tournament_matches_played',
    'team_b_tournament_matches_played',
]

new_rows = model_df[model_df['date'] > CUTOFF_DATE].copy()
all_rows = model_df.copy()

print(f'New rows (post {CUTOFF_DATE.date()}): {len(new_rows)}')
print(f'All rows: {len(all_rows):,}')
print(f'Features inspected: {len(FEATURE_COLS_20)}')

New rows (post 2026-05-16): 119
All rows: 21,655
Features inspected: 20


In [44]:
# ── Per-feature summary table ──────────────────────────────────────────────────
rows = []
for col in FEATURE_COLS_20:
    s_new = new_rows[col]
    s_all = all_rows[col]
    rows.append({
        'feature':       col,
        'zeros_new':     int((s_new == 0).sum()),
        'zeros_new_%':   round((s_new == 0).mean() * 100, 1),
        'mean_new':      round(s_new.mean(), 3),
        'std_new':       round(s_new.std(), 3),
        'min_new':       round(s_new.min(), 3),
        'max_new':       round(s_new.max(), 3),
        'mean_all':      round(s_all.mean(), 3),
    })

summary = pd.DataFrame(rows).set_index('feature')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print('Feature summary — NEW rows only (post May 16 2026)')
print('zeros_new = exact zeros | zeros_new_% = % of new rows that are 0 | mean_all = mean across full dataset for comparison\n')
display(summary)

Feature summary — NEW rows only (post May 16 2026)
zeros_new = exact zeros | zeros_new_% = % of new rows that are 0 | mean_all = mean across full dataset for comparison



,zeros_new,zeros_new_%,mean_new,std_new,min_new,max_new,mean_all
feature,,,,,,,
rank_diff,3,2.5,-28.034,41.712,-134.000,77.000,-12.199
elo_diff,0,0.0,148.345,221.512,-420.000,633.000,52.251
rating_a_before,0,0.0,1568.622,341.470,633.000,2165.000,1506.930
rating_b_before,0,0.0,1420.277,316.813,631.000,1925.000,1454.679
avg_player_value_diff,2,1.7,4.148,10.694,-25.460,49.320,0.641
opponent_strength_diff_last5,0,0.0,94.010,207.770,-539.200,703.400,17.923
weighted_goals_for_diff_last5,0,0.0,0.311,0.816,-1.633,2.390,0.101
weighted_goals_against_diff_last5,0,0.0,-0.286,1.042,-4.067,2.031,-0.160
team_b_matches_played_before,0,0.0,212.807,77.529,50.000,359.000,109.917


In [45]:
# ── Flag suspicious features (>50% zeros or very different mean vs full dataset) ──
print('=== SUSPICIOUS FEATURES ===')
flagged = False
for col in FEATURE_COLS_20:
    s_new = new_rows[col]
    s_all = all_rows[col]
    zero_pct   = (s_new == 0).mean() * 100
    mean_new   = s_new.mean()
    mean_all   = s_all.mean()
    std_all    = s_all.std()
    # Flag if >50% zeros OR mean is more than 2 std deviations away from full-dataset mean
    z_score    = abs(mean_new - mean_all) / (std_all + 1e-9)
    if zero_pct > 50 or z_score > 2:
        reason = []
        if zero_pct > 50:
            reason.append(f'{zero_pct:.0f}% zeros')
        if z_score > 2:
            reason.append(f'mean_new={mean_new:.3f} vs mean_all={mean_all:.3f} (z={z_score:.1f})')
        print(f'  ⚠  {col:<45} {", ".join(reason)}')
        flagged = True
if not flagged:
    print('  None — all features look consistent.')

=== SUSPICIOUS FEATURES ===
  None — all features look consistent.


In [46]:
# ── Deep dive: per-feature value counts for zero-heavy or suspicious features ──
for col in FEATURE_COLS_20:
    zero_pct = (new_rows[col] == 0).mean() * 100
    if zero_pct > 30:
        print(f'\n{col}  ({zero_pct:.0f}% zeros in new rows)')
        nonzero = new_rows[col][new_rows[col] != 0]
        print(f'  Non-zero count: {len(nonzero)}  |  mean: {nonzero.mean():.4f}  |  '
              f'min: {nonzero.min():.4f}  max: {nonzero.max():.4f}')
        # Show which competitions contribute the zeros
        zero_rows = new_rows[new_rows[col] == 0]
        print(f'  Zero rows competition breakdown:')
        print('  ' + str(zero_rows['competition'].value_counts().head(5)).replace('\n', '\n  '))


tournament_points_diff  (43% zeros in new rows)
  Non-zero count: 68  |  mean: 1.4265  |  min: -7.0000  max: 14.0000
  Zero rows competition breakdown:
  competition
  Friendly                      36
  Unity Cup                      4
  Diamond Jubilee Tournament     3
  Friendly tournament            2
  Garuda Championship Series     2
  Name: count, dtype: int64

team_b_tournament_matches_played  (39% zeros in new rows)
  Non-zero count: 72  |  mean: 1.8333  |  min: 1.0000  max: 6.0000
  Zero rows competition breakdown:
  competition
  Friendly                      36
  Unity Cup                      2
  Diamond Jubilee Tournament     2
  Garuda Championship Series     2
  Baltic Cup                     2
  Name: count, dtype: int64


In [47]:
# ── Sample rows: show 10 new rows with all 20 features ───────────────────────
print('=== SAMPLE NEW ROWS (10 random) — all 20 features ===\n')
sample_new = new_rows[['date','team_a','team_b','competition'] + FEATURE_COLS_20].sample(
    min(10, len(new_rows)), random_state=42
).sort_values('date')

# Print each row as a column-by-column summary so it's readable
for _, r in sample_new.iterrows():
    print(f"{r['date'].date()}  {r['team_a']} vs {r['team_b']}  [{r['competition']}]")
    for col in FEATURE_COLS_20:
        val = r[col]
        flag = ' ← ZERO' if val == 0 and col in ['goalkeeper_share_diff','defender_share_diff',
                                                   'avg_player_value_diff','market_value_rel_mean_diff'] else ''
        print(f'  {col:<45} {val:>10.4f}{flag}')
    print()

=== SAMPLE NEW ROWS (10 random) — all 20 features ===

2026-05-28  Ireland vs Qatar  [Friendly]
  rank_diff                                       -48.0000
  elo_diff                                        267.0000
  rating_a_before                                1692.0000
  rating_b_before                                1425.0000
  avg_player_value_diff                            -0.7300
  opponent_strength_diff_last5                    143.6000
  weighted_goals_for_diff_last5                     1.6522
  weighted_goals_against_diff_last5                -0.9022
  team_b_matches_played_before                    358.0000
  team_a_matches_played_before                    236.0000
  market_value_rel_mean_diff                       -0.1862
  rating_change_diff_last5                         28.6000
  defender_share_diff                              -0.1233
  goalkeeper_share_diff                            -0.0904
  team_b_days_since_last_match                     60.0000
  team_a_days_since

In [42]:
model_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved → {OUTPUT_PATH}')
print(f'Shape : {model_df.shape}')
print(f'Columns: {list(model_df.columns)}')

Saved → ..\data\processed\updated_model_dataset.csv
Shape : (21655, 33)
Columns: ['date', 'team_a', 'team_b', 'competition', 'location', 'season_id', 'tournament_year', 'tournament_key', 'rank_diff', 'elo_diff', 'rating_a_before', 'rating_b_before', 'avg_player_value_diff', 'log_market_value_a', 'market_value_rel_mean_diff', 'opponent_strength_diff_last5', 'weighted_goals_for_diff_last5', 'weighted_goals_against_diff_last5', 'rating_change_diff_last5', 'team_a_matches_played_before', 'team_b_matches_played_before', 'team_a_days_since_last_match', 'team_b_days_since_last_match', 'team_a_tournament_matches_played', 'team_b_tournament_matches_played', 'tournament_points_diff', 'tournament_goal_diff_diff', 'goalkeeper_share_diff', 'defender_share_diff', 'target_goals_a', 'target_goals_b', 'target_goal_diff', 'target_total_goals']
